# Data Extraction
The sales files are arranged strangely as usual, with multiple files per year, and sometime with checkpoints.  
I'm going to follow the same strategy as Gwinnett, where I just take all of the sales data and deduplicate at the end.  

**Selected Data Files**  
These were all taken from the Dekalb-Hicks folder of the County Tax Assessment Data Folder  
Dekalb Count Full File2015/SALES.CSV  
Dekalb Count Full File EOY2015/SALES.CSV  
Dekalb Count Full File EOY2016/SALES.CSV  
Dekalb Count Full File EOY2017/SALES.CSV  
Dekalb Count Full File Oct 2018/SALES.CSV  
Dekalb Count Full File EOY2019/SALES.CSV  
Dekalb Count Full File EOY2020/SALES.CSV  
Dekalb Count Full File Aug 2021/SALES.CSV  

They were all renamed to the format SALES_20{XX}.CSV for convenience

In [1]:
import pandas as pd
import os

DATA_PATH = "../../data/dekalb"
OUT_PATH = "../../data/dekalb/out"

In [2]:
year_dfs = []

for file_p in os.listdir(DATA_PATH):
    if file_p.endswith(".CSV"):
        year = int(file_p.split("_")[1].split(".")[0])
        df = pd.read_csv(os.path.join(DATA_PATH, file_p))

        year_dfs.append((year, df))

In [3]:
year_dfs.sort(key = lambda x : x[0])

In [4]:
for year, df in year_dfs:
    print(year)
    print(df.columns)

2015
Index(['PARID', ' BOOK', ' PAGE', ' OLDOWN', ' OWN1', ' SALEDT', ' PRICE',
       ' SALETYPE', ' INSTRTYP', ' SALEVALDESCR'],
      dtype='object')
2016
Index(['PARID', ' BOOK', ' PAGE', ' OLDOWN', ' OWN1', ' SALEDT', ' PRICE',
       ' SALETYPE', ' INSTRTYP', ' SALEVALDESCR'],
      dtype='object')
2017
Index(['PARID', ' BOOK', ' PAGE', ' OLDOWN', ' OWN1', ' SALEDT', ' PRICE',
       ' SALETYPE', ' INSTRTYP', ' SALEVALDESCR'],
      dtype='object')
2018
Index(['PARID', ' BOOK', ' PAGE', ' OLDOWN', ' OWN1', ' SALEDT', ' PRICE',
       ' SALETYPE', ' INSTRTYP', ' SALEVALDESCR'],
      dtype='object')
2019
Index(['PARID', ' BOOK', ' PAGE', ' OLDOWN', ' OWN1', ' SALEDT', ' PRICE',
       ' SALETYPE', ' INSTRTYP', ' SALEVALDESCR'],
      dtype='object')
2020
Index(['PARID', ' BOOK', ' PAGE', ' OLDOWN', ' OWN1', ' SALEDT', ' PRICE',
       ' SALETYPE', ' INSTRTYP', ' SALEVALDESCR'],
      dtype='object')
2021
Index(['PARID', ' BOOK', ' PAGE', ' OLDOWN', ' OWN1', ' SALEDT', ' PRI

Luckily looks like the entire thing has the same data format.

In [ ]:
year_df_list = []
for year, year_df in year_dfs:
    year_df = year_df.rename(columns={old: new for old, new in zip(year_df.columns, year_df.columns.str.strip())})

    for column in year_df.columns:
        if year_df.dtypes[column] == object:
            year_df[column] = year_df[column].str.strip()
    
    year_df['SALEDT'] = pd.to_datetime(year_df['SALEDT'], format="%d-%b-%y" ,errors="coerce")
    year_df['SALE_YR'] = year_df["SALEDT"].dt.year
    year_df_list.append(year_df)

total_digest_df = pd.concat(year_df_list)

In [13]:
old_rows = total_digest_df.shape[0]
total_digest_df_dedup = total_digest_df.drop_duplicates(subset=['PARID', 'BOOK', 'PAGE', 'SALEDT'])

new_rows = total_digest_df_dedup.shape[0]

print(f"{old_rows - new_rows} rows removed. {new_rows} Remaining.")

85 rows removed. 112149 Remaining.


In [14]:
total_digest_df_dedup = total_digest_df_dedup.sort_values(by = "SALEDT")

In [15]:
total_digest_df_dedup.to_csv(os.path.join(OUT_PATH, "DEKALB_SALES_FINAL.csv"), index=False)

In [9]:
tax_digest = pd.read_csv(os.path.join(DATA_PATH, "dekalb_digest_withnonprofit.csv"))

/var/folders/bb/g7vlcgfn14ld8_w41331g0dw0000gn/T/ipykernel_79855/2897194920.py:1: DtypeWarning: Columns (4,5,6,7,8,14,15,25,45,46,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,67,68,69,70,71,72,73,74,75,76,77,78,79,80,82,83,86,87,88,89,90,97,98,100) have mixed types. Specify dtype option on import or set low_memory=False.
  tax_digest = pd.read_csv(os.path.join(DATA_PATH, "dekalb_digest_withnonprofit.csv"))


In [11]:
tax_digest.columns.to_list()

['PARID',
 'TAXYR',
 'OWN1',
 'OWN2',
 'ADRSTR_digest',
 'CITYNAME_digest',
 'ZIP1_digest',
 'ZIP2_digest',
 'PCTOWN',
 'ASMT_TAXYR',
 'ASMT_FMV_LAND',
 'ASMT_FMV_BLDG',
 'LAND_TAXYR',
 'LAND_LLINE',
 'LAND_LTYPE',
 'LAND_CODE',
 'LAND_SF',
 'LAND_ACRES',
 'LAND_UNITS',
 'LAND_BRATE',
 'LAND_INFL1',
 'LAND_INFL2',
 'LAND_INFLU',
 'LAND_PRICE',
 'LAND_ADJFACT',
 'LAND_CLASS',
 'LAND_INFLU2',
 'DWELL_CARD',
 'DWELL_TAXYR',
 'DWELL_STORIES',
 'DWELL_EXTWALL',
 'DWELL_STYLE',
 'DWELL_YRBLT',
 'DWELL_RMBED',
 'DWELL_FIXBATH',
 'DWELL_FIXHALF',
 'DWELL_FIXADDL',
 'DWELL_FIXTOT',
 'DWELL_BSMT',
 'DWELL_HEAT',
 'DWELL_FINBSMTAREA',
 'DWELL_WBFP_O',
 'DWELL_WBFP_S',
 'DWELL_WBFP_PF',
 'DWELL_BSMTCAR',
 'DWELL_GRADE',
 'DWELL_CDU',
 'DWELL_SFLA',
 'NBHD',
 'ADRNO',
 'ADRADD',
 'ADRDIR',
 'ADRSTR_owner',
 'ADRSUF',
 'ADRSUF2',
 'CITYNAME_owner',
 'STATECODE',
 'COUNTRY',
 'POSTALCODE',
 'UNITNO',
 'ADDR1',
 'ADDR2',
 'ADDR3',
 'ZIP1_owner',
 'ZIP2_owner',
 'full_owner_name',
 'owner_type',
 'mod_

In [16]:
sales_tax_merged = pd.merge(total_digest_df_dedup, tax_digest, how="left", left_on=["PARID", "SALE_YR"], right_on=["PARID", "TAXYR"])

In [20]:
pct_merged = sales_tax_merged[sales_tax_merged["TAXYR"].notna()].shape[0] / sales_tax_merged.shape[0]
print(f"Pct Succesfully Merged: {pct_merged}")

Pct Succesfully Merged: 0.9942576393904538


In [22]:
sales_tax_merged[sales_tax_merged["TAXYR"].isna()]["PARID"].unique()

array(['16 010 01 218', '16 010 01 195', '15 209 03 187', '16 010 01 220',
       '18 168 05 011', '18 345 01 344', '18 345 01 345', '18 146 02 165',
       '18 345 01 305', '18 345 01 304', '18 345 01 318', '18 345 01 346',
       '18 343 13 002', '16 010 01 221', '18 046 01 169', '18 046 02 060',
       '18 046 02 059', '18 046 02 058', '18 046 01 170', '18 146 01 129',
       '18 345 01 341', '18 146 02 164', '18 065 12 051', '18 371 01 026',
       '16 010 01 205', '18 146 02 166', '18 262 06 069', '15 084 01 158',
       '15 249 04 035', '18 345 01 340', '16 010 01 193', '18 146 02 177',
       '18 238 10 024', '18 238 10 025', '18 238 10 026', '18 074 03 164',
       '16 010 01 225', '15 121 06 017', '15 177 03 016', '15 177 03 019',
       '15 241 01 202', '16 010 01 208', '16 120 02 010', '18 332 03 072',
       '18 332 03 069', '18 332 03 074', '18 332 03 070', '18 332 03 073',
       '18 332 03 071', '16 124 02 016', '16 010 01 197', '06 311 02 071',
       '16 010 01 192', '

In [23]:
sales_tax_merged.to_csv(os.path.join(OUT_PATH, "DEKALB_SALES_TAX_DIGEST_FINAL.csv"), index=False)

In [ ]:
tax_digest.shape

(2384941, 104)

: 